In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from collections import Counter

In [ ]:
train_dir = "../data/train"
test_dir = "../data/test"

In [ ]:
os.listdir(train_dir)

In [ ]:
class_counts = {}

for emotion in os.listdir(train_dir):
    
    emotion_path = os.path.join(train_dir, emotion)

    if os.path.isdir(emotion_path):
        class_counts[emotion] = len(os.listdir(emotion_path))

class_counts

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    x=list(class_counts.keys()),
    y=list(class_counts.values())
)

plt.title("Class Distribution in FER2013")
plt.xlabel("Emotion")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

axes = axes.flatten()

for idx, emotion in enumerate(os.listdir(train_dir)[:7]):

    emotion_path = os.path.join(train_dir, emotion)

    image_name = random.choice(os.listdir(emotion_path))

    image_path = os.path.join(emotion_path, image_name)

    image = Image.open(image_path)

    axes[idx].imshow(image, cmap="gray")
    axes[idx].set_title(emotion)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sample_emotion = random.choice(os.listdir(train_dir))

sample_path = os.path.join(
    train_dir,
    sample_emotion,
    random.choice(os.listdir(os.path.join(train_dir, sample_emotion)))
)

image = Image.open(sample_path)

image_array = np.array(image)

print("Image shape:", image_array.shape)

In [ ]:
print("Minimum pixel value:", image_array.min())
print("Maximum pixel value:", image_array.max())

In [ ]:
plt.figure(figsize=(4,4))

plt.imshow(image_array, cmap="gray")

plt.title(f"Emotion: {sample_emotion}")

plt.colorbar()

plt.show()

# EDA Observations

## Dataset Findings

- FER2013 contains grayscale facial images of size 48×48.
- The dataset is imbalanced, especially for the disgust class.
- Several images are blurry, noisy, and low contrast.
- Some emotions visually overlap, especially fear and surprise.
- Small image resolution makes emotion recognition difficult.
- Data augmentation and regularization will likely be important.

# Phase 2 — Data Pipeline

In [ ]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor()
])


In [ ]:
from PIL import Image
image_path = "../data/train/happy/Training_1866804.jpg"

image = Image.open(image_path)

image

tensor_image = transform(image)

tensor_image.shape

We imported one image manually

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(
    tensor_image.squeeze(),
    cmap="gray"
)

plt.show()

In [ ]:
transform = transforms.Compose([
    
    transforms.Resize((48, 48)),
    
    transforms.RandomHorizontalFlip(),
    
    transforms.RandomRotation(10),
    
    transforms.ToTensor()
])

Rotation and Horizantal Flip transformation

In [ ]:
tensor_image = transform(image)

plt.imshow(
    tensor_image.squeeze(),
    cmap="gray"
)

plt.axis("off")

plt.show()

In [ ]:
transform = transforms.Compose([
    
    transforms.Resize((48, 48)),
    
    transforms.RandomHorizontalFlip(),
    
    transforms.RandomRotation(10),
    
    transforms.ColorJitter(
        brightness=0.2
    ),
    
    transforms.ToTensor()
])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12,4))

for i in range(4):
    
    aug_img = transform(image)
    
    axes[i].imshow(
        aug_img.squeeze(),
        cmap="gray"
    )
    
    axes[i].axis("off")

plt.show()

In [ ]:
from torchvision import datasets
train_dataset = datasets.ImageFolder(
    root="../data/train",
    transform=transform
)

In [ ]:
train_dataset
train_dataset.class_to_idx

In [ ]:
image, label = train_dataset[0]

In [ ]:
print(image.shape)
print(label)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 64

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [ ]:
images, labels = next(iter(train_loader))

In [ ]:
print(images.shape)
print(labels.shape)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10,5))

axes = axes.flatten()

for i in range(8):

    axes[i].imshow(
        images[i].permute(1,2,0)
    )

    axes[i].set_title(labels[i].item())

    axes[i].axis("off")

plt.tight_layout()

plt.show()

In [ ]:
idx_to_class = {
    v:k for k,v in train_dataset.class_to_idx.items()
}

In [ ]:
transform = transforms.Compose([
    
    transforms.Grayscale(num_output_channels=1),
    
    transforms.Resize((48, 48)),
    
    transforms.RandomHorizontalFlip(),
    
    transforms.RandomRotation(10),
    
    transforms.ColorJitter(
        brightness=0.2
    ),
    
    transforms.ToTensor()
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root="../data/train",
    transform=transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [ ]:
images, labels = next(iter(train_loader))

In [ ]:
print(images.shape)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10,5))

axes = axes.flatten()

for i in range(8):

    axes[i].imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    label_name = idx_to_class[
        labels[i].item()
    ]

    axes[i].set_title(label_name)

    axes[i].axis("off")

plt.tight_layout()

plt.show()

In [ ]:
from torch.utils.data import WeightedRandomSampler

In [ ]:
targets = train_dataset.targets

print(targets[:10])

In [ ]:
class_counts = np.bincount(targets)

print(class_counts)

In [ ]:
class_weights = 1. / class_counts

print(class_weights)

In [ ]:
sample_weights = [
    class_weights[label]
    for label in targets
]

In [ ]:
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
transform = transforms.Compose([
    
    transforms.Grayscale(num_output_channels=1),

    transforms.Resize((48, 48)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root="../data/train",
    transform=transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

In [ ]:
images, labels = next(iter(train_loader))

In [ ]:
print(images.min())
print(images.max())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10,5))

axes = axes.flatten()

for i in range(8):

    axes[i].imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    label_name = idx_to_class[
        labels[i].item()
    ]

    axes[i].set_title(label_name)

    axes[i].axis("off")

plt.tight_layout()

plt.show()